# SalesPrep — exploration

Parcours complet des étapes 1.a à 7 : chargement ventes, agrégations, imputations, pivots et jointure.

Remplit `../Output/` avec chaque dataframe intermédiaire.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT.name != "SalesPrep" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
PREPARE = ROOT.parent
PROJECT = PREPARE.parent

sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(ROOT / "Src"))

INPUT_DIR = ROOT / "Input"
OUTPUT_DIR = ROOT / "Output"
ROD_OUTPUT = PREPARE / "RodPrep" / "Output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

## 1. Entrées — ventes brutes et lookup hôtel

In [ ]:
from rod_ia.config.settings import get_settings
from prepare._shared.sales_base import load_sales_frame

settings = get_settings(PROJECT)
sales_path = settings.sales_csv_path
holdout_year = 2026

sales_raw_head = pd.read_csv(sales_path, nrows=8)
print("Aperçu fichier ventes (8 lignes) :")
sales_raw_head

In [ ]:
lookup_path = ROD_OUTPUT / "hotel_lookup.parquet"
rod_lookup = pd.read_parquet(lookup_path) if lookup_path.exists() else None
if rod_lookup is not None:
    rod_lookup = rod_lookup[["nom_hotel", "hotel_code"]].drop_duplicates()
    print(f"Lookup RodPrep : {len(rod_lookup)} hôtels")
    rod_lookup.head()
else:
    print("Exécuter d'abord RodPrep/Explore/explore.ipynb")

## 2. Phase 0 — normalisation ligne à ligne

In [ ]:
raw = load_sales_frame(sales_path, exclude_year=holdout_year)
if rod_lookup is not None:
    raw = raw.merge(rod_lookup, on="nom_hotel", how="left")
    raw["hotel_code"] = raw["hotel_code"].fillna(raw["nom_hotel"])
else:
    raw["hotel_code"] = raw["nom_hotel"]

print(f"Lignes retenues (annee < {holdout_year}) : {len(raw):,}")
raw[["nom_hotel", "hotel_code", "annee", "mois", "categorie", "sous_categorie",
     "nombre_ventes", "montant_ventes", "heure_vente", "is_weekend", "is_holiday"]].head(8)

## 3. Étape 1 — agrégation annuelle

In [ ]:
from sales_prep import aggregations as agg

step_1a = agg.step_1a_annual_raw(raw)
step_1b = agg.step_1b_annual_normalized(step_1a)
step_1c = agg.step_1c_annual_divided_by_12(step_1a)

print("1.a — brut annuel")
step_1a.head()

In [ ]:
print("1.b — annualisé (÷ mois_actifs × 12)")
step_1b.head()

In [ ]:
print("1.c — divisé par 12")
step_1c.head()

## 4. Étape 2 — agrégation mensuelle + imputation

In [ ]:
step_2a = agg.step_2a_monthly_raw(raw)
step_2b = agg.step_2b_monthly_imputed(step_2a)

print(f"2.a : {len(step_2a)} lignes | 2.b : {len(step_2b)} lignes (après imputation)")
step_2a.head(8)

In [ ]:
step_2b.sort_values(["nom_hotel", "annee", "mois"]).head(12)

## 5. Étape 3 — catégorie / sous-catégorie

In [ ]:
step_3a = agg.step_3a_category_monthly(raw)
step_3b = agg.step_3b_category_imputed(step_3a, step_2a)
step_3c = agg.step_3c_category_wide(step_3b)

print(f"3.a: {step_3a.shape} | 3.b: {step_3b.shape} | 3.c: {step_3c.shape}")
step_3a.head(8)

In [ ]:
step_3c.head()

## 6. Étape 4 — heure de vente

In [ ]:
step_4a = agg.step_4a_hourly(step_3b, raw)
step_4b = agg.step_4b_hourly_imputed(step_4a, step_2a)
step_4c = agg.step_4c_hourly_wide(step_4b)
print(f"4.a: {step_4a.shape} | 4.b: {step_4b.shape} | 4.c: {step_4c.shape}")
step_4a.head(8)

## 7. Étape 5 — week-end

In [ ]:
step_5a = agg.step_5a_weekend(raw)
step_5b = agg.step_5b_weekend_imputed(step_5a, step_2a)
step_5c = agg.step_5c_weekend_wide(step_5b)
print(f"5.a: {step_5a.shape} | 5.c: {step_5c.shape}")
step_5a.head(8)

## 8. Étape 6 — jour férié

In [ ]:
step_6a = agg.step_6a_holiday(raw)
step_6b = agg.step_6b_holiday_imputed(step_6a, step_2a)
step_6c = agg.step_6c_holiday_wide(step_6b)
print(f"6.a: {step_6a.shape} | 6.c: {step_6c.shape}")
step_6a.head(8)

## 9. Étape 7 — jointure finale

In [ ]:
from sales_prep.pipeline import SalesPrep

joined = SalesPrep._join_all(
    [step_2b, step_3c, step_4c, step_5c, step_6c],
    keys=["nom_hotel", "hotel_code", "annee", "mois"],
)
print(f"Jointure : {joined.shape}")
joined.head()

## 10. Persistance Output/

In [ ]:
artifacts = {
    "step_1a": step_1a, "step_1b": step_1b, "step_1c": step_1c,
    "step_2a": step_2a, "step_2b": step_2b,
    "step_3a": step_3a, "step_3b": step_3b, "step_3c": step_3c,
    "step_4a": step_4a, "step_4b": step_4b, "step_4c": step_4c,
    "step_5a": step_5a, "step_5b": step_5b, "step_5c": step_5c,
    "step_6a": step_6a, "step_6b": step_6b, "step_6c": step_6c,
    "joined": joined,
}

for name, df in artifacts.items():
    df.to_parquet(OUTPUT_DIR / f"{name}.parquet", index=False)
    df.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)
    print(f"  {name} → {df.shape}")

print("\nTerminé — fichiers dans", OUTPUT_DIR)